In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import matplotlib.pyplot as plt
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
import pyarrow as pa


THRESHOLD_TIMESTAMPS = 15

In [3]:
df_merged = dd.read_parquet("merged_dataset.parquet")
print(len(df_merged))

22298361


In [ ]:

df_merged["page_visit_timestamp"].head().apply(type)

In [4]:
list_cols_data_set = ["page_visit_timestamp", "add_to_cart_timestamp", "product_buy_timestamp",
                      "remove_from_cart_timestamp", "search_query_timestamp"] 

In [ ]:
df_merged[list_cols_data_set] = df_merged[list_cols_data_set].map_partitions(
    lambda pdf: pdf.apply(
        lambda col: col.apply(
            lambda x: x.tolist() if isinstance(x, np.ndarray) else ([] if x is None else x)
        )
    )
)


In [6]:
df_merged["Total_timestamps"] = df_merged[list_cols_data_set].map_partitions(
    lambda pdf: pdf.apply(
        lambda row: sum(len(x) if isinstance(x, list) else 0 for x in row),
        axis=1
    )
)

In [ ]:
df_merged["Total_timestamps"].tail()

client_id
23929214     2
23929215     1
23929216     3
23929217     3
23929218    75
Name: Total_timestamps, dtype: int64

In [ ]:
with ProgressBar():
    list_ts = df_merged["Total_timestamps"].compute().tolist()

[#####                                   ] | 13% Completed | 16.30 ss


In [15]:
print(np.median(list_ts))
print(len(list_ts))

2.0
22298361


In [8]:
with ProgressBar():
    user_with_few_events = (df_merged["Total_timestamps"].isin([1,2,3,4,5])).sum().compute()
    total_users = df_merged.shape[0].compute()
    percent_users_few_events = (user_with_few_events / total_users) * 100
    
    print("User Activity")
    print(f"Total users: {total_users}")
    print(f"Users with 1–5 events: {user_with_few_events}")
    print(f"Percentage of users with 1–5 events: {percent_users_few_events:.2f}%")

[#############################           ] | 72% Completed | 231.40 s

IOStream.flush timed out


[########################################] | 100% Completed | 394.41 s
[########################################] | 100% Completed | 103.34 ms
User Activity
Total users: 22298361
Users with 1–5 events: 16101834
Percentage of users with 1–5 events: 72.21%


In [ ]:
with ProgressBar():
    passed_lenghts_timestamps = df_merged[df_merged["Total_timestamps"] > THRESHOLD_TIMESTAMPS]["Total_timestamps"].compute()
    print(f"Total after the threshold {len(passed_lenghts_timestamps)}")
    print(f"Median after the threshold {np.median(passed_lenghts_timestamps)}")

In [ ]:
total_sessions = len(df_merged)
kept_sessions = len(passed_lenghts_timestamps)
lost_sessions = total_sessions - kept_sessions

percent_lost = (lost_sessions / total_sessions) * 100
print(f"{percent_lost:.2f}%")